# Advanced CVE Prioritization with Graph Neural Networks

**Date:** January 27, 2026  
**Phase:** 7 - Advanced Models  
**Objective:** Implement and evaluate graph-based models for CVE prioritization

---

## Overview

This notebook explores advanced machine learning models for CVE prioritization:

1. **DiffusionRank**: Random walk-based algorithm on vulnerability similarity graph
2. **RGCN**: Relational Graph Convolutional Network modeling CVE-CWE relationships  
3. **Ensemble**: Meta-learner combining multiple model predictions

### Why Graph Models?

Traditional ML models (LambdaRank, XGBoost) treat CVEs as independent entities. Graph models capture:
- **Structural relationships**: CVE → CWE → Other CVEs
- **Similarity propagation**: High-priority CVEs influence similar vulnerabilities
- **Network effects**: Exploited CVEs in same CWE category are more risky

### Expected Improvements

- **NDCG@20**: 0.95 → 0.97 (+2% lift)
- **Healthcare Recall**: Better identification of medical device CVEs
- **Explainability**: Graph paths show why CVEs are related

In [ ]:
# Core imports
import sys
import sqlite3
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Graph libraries
import networkx as nx
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

# Local imports
sys.path.append('..')
from src.core.database import CVEDatabase
from src.features.engineering import create_all_features
from src.evaluation.metrics import ndcg_at_k, precision_at_k

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ Imports successful")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NetworkX: {nx.__version__}")
print(f"LightGBM: {lgb.__version__}")

## 1. Data Loading & Preprocessing

Load CVE data with enrichments and extract graph structure.

In [ ]:
# Connect to database
db_path = Path('../data/cve_database.db')
db = CVEDatabase(str(db_path))

print(f"Database: {db_path}")
print(f"Total CVEs: {db.count_cves():,}")

In [ ]:
# Load CVE data with enrichments
query = """
SELECT 
    c.cve_id,
    c.published,
    c.cvss,
    c.cwe,
    e.epss_score,
    e.epss_percentile,
    e.kev_flag,
    e.is_healthcare,
    e.attack_flag,
    e.attack_technique_count,
    e.chpl_flag,
    e.label
FROM cves c
LEFT JOIN enrichments e ON c.cve_id = e.cve_id
WHERE c.published >= '2024-01-01'
    AND c.cvss IS NOT NULL
    AND e.label IS NOT NULL
ORDER BY c.published DESC
"""

df = pd.read_sql(query, db.conn)
df['published'] = pd.to_datetime(df['published'])

print(f"Loaded CVEs: {len(df):,}")
print(f"Date range: {df['published'].min()} to {df['published'].max()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())

df.head()

In [ ]:
# Extract CWE information
df['has_cwe'] = df['cwe'].notna() & (df['cwe'] != '')
df['cwe_primary'] = df['cwe'].str.extract(r'(CWE-\d+)')[0]  # Extract first CWE

print(f"CVEs with CWE: {df['has_cwe'].sum():,} ({df['has_cwe'].mean()*100:.1f}%)")
print(f"Unique CWEs: {df['cwe_primary'].nunique():,}")
print(f"\nTop 10 CWEs:")
print(df['cwe_primary'].value_counts().head(10))

In [ ]:
# Temporal split for validation
split_date = pd.Timestamp('2024-11-01')

train_df = df[df['published'] < split_date].copy()
test_df = df[df['published'] >= split_date].copy()

print(f"Training set: {len(train_df):,} CVEs (before {split_date.date()})")
print(f"Test set: {len(test_df):,} CVEs (from {split_date.date()})")
print(f"\nTrain label distribution:")
print(train_df['label'].value_counts(normalize=True).round(3))
print(f"\nTest label distribution:")
print(test_df['label'].value_counts(normalize=True).round(3))

## 2. Baseline: LambdaRank Model

Load and evaluate the existing LambdaRank model as our baseline.

In [ ]:
# Load pre-trained model
import pickle

model_path = Path('../models/ltr_model_conf_weighted.pkl')
with open(model_path, 'rb') as f:
    baseline_model = pickle.load(f)

print(f"✓ Loaded baseline model: {model_path.name}")
print(f"Model type: {type(baseline_model).__name__}")

In [ ]:
# Feature engineering for baseline
feature_cols = [
    'cvss_norm', 'epss_score', 'epss_percentile', 'kev_flag',
    'recency_score', 'attack_technique_count', 'has_attack',
    'chpl_flag', 'is_healthcare', 'cvss_epss_product',
    'kev_healthcare_interaction'
]

train_features = create_all_features(train_df, feature_cols)
test_features = create_all_features(test_df, feature_cols)

X_train = train_features[feature_cols]
y_train = train_features['label']
X_test = test_features[feature_cols]
y_test = test_features['label']

print(f"Training features: {X_train.shape}")
print(f"Test features: {X_test.shape}")

In [ ]:
# Baseline predictions
baseline_scores = baseline_model.predict(X_test)
test_df['baseline_score'] = baseline_scores

# Evaluate baseline
from src.evaluation.metrics import ndcg_at_k, precision_at_k

k_values = [10, 20, 50, 100]
baseline_results = {}

for k in k_values:
    ndcg = ndcg_at_k(y_test, baseline_scores, k)
    prec = precision_at_k(y_test, baseline_scores, k, threshold=3)
    baseline_results[f'NDCG@{k}'] = ndcg
    baseline_results[f'Precision@{k}'] = prec

print("Baseline LambdaRank Performance:")
print("=" * 40)
for metric, value in baseline_results.items():
    print(f"{metric:20s}: {value:.4f}")

# Healthcare-specific metrics
healthcare_test = test_df[test_df['is_healthcare'] == 1]
if len(healthcare_test) > 0:
    top_100_healthcare = test_df.nlargest(100, 'baseline_score')['is_healthcare'].sum()
    print(f"\nHealthcare CVEs in Top-100: {top_100_healthcare}")
    print(f"Healthcare Recall@100: {top_100_healthcare / len(healthcare_test):.2%}")

## 3. Graph Construction

Build two types of graphs:
1. **CVE-CWE Bipartite Graph**: Connects CVEs to their weakness types
2. **CVE Similarity Graph**: Connects similar CVEs based on features

In [ ]:
# 3.1: CVE-CWE Bipartite Graph
G_bipartite = nx.Graph()

# Add CVE nodes
for idx, row in train_df.iterrows():
    G_bipartite.add_node(
        row['cve_id'],
        node_type='cve',
        cvss=row['cvss'],
        epss=row['epss_score'],
        kev=row['kev_flag'],
        label=row['label']
    )

# Add CWE nodes and edges
cve_cwe_edges = 0
for idx, row in train_df[train_df['has_cwe']].iterrows():
    cwe = row['cwe_primary']
    if cwe:
        if cwe not in G_bipartite:
            G_bipartite.add_node(cwe, node_type='cwe')
        G_bipartite.add_edge(row['cve_id'], cwe, edge_type='has_weakness')
        cve_cwe_edges += 1

print(f"Bipartite Graph:")
print(f"  CVE nodes: {sum(1 for n, d in G_bipartite.nodes(data=True) if d.get('node_type') == 'cve'):,}")
print(f"  CWE nodes: {sum(1 for n, d in G_bipartite.nodes(data=True) if d.get('node_type') == 'cwe'):,}")
print(f"  Edges: {G_bipartite.number_of_edges():,}")
print(f"  Density: {nx.density(G_bipartite):.6f}")

In [ ]:
# 3.2: CVE Similarity Graph
# Create feature vectors for similarity
similarity_features = ['cvss', 'epss_score', 'kev_flag', 'is_healthcare', 'attack_technique_count']
feature_matrix = train_df[similarity_features].fillna(0).values

# Normalize features
scaler = StandardScaler()
feature_matrix_normalized = scaler.fit_transform(feature_matrix)

# Compute cosine similarity
similarity_matrix = cosine_similarity(feature_matrix_normalized)

# Create similarity graph (keep only top-k similar CVEs per node)
k_neighbors = 10
similarity_threshold = 0.7

G_similarity = nx.Graph()
G_similarity.add_nodes_from(train_df['cve_id'])

edges_added = 0
for i, cve_i in enumerate(train_df['cve_id']):
    # Get top-k most similar CVEs
    similarities = similarity_matrix[i]
    top_k_indices = np.argsort(similarities)[::-1][1:k_neighbors+1]  # Exclude self
    
    for j in top_k_indices:
        if similarities[j] >= similarity_threshold:
            cve_j = train_df.iloc[j]['cve_id']
            G_similarity.add_edge(cve_i, cve_j, weight=similarities[j])
            edges_added += 1

print(f"\nSimilarity Graph:")
print(f"  Nodes: {G_similarity.number_of_nodes():,}")
print(f"  Edges: {G_similarity.number_of_edges():,}")
print(f"  Avg degree: {sum(dict(G_similarity.degree()).values()) / G_similarity.number_of_nodes():.2f}")
print(f"  Connected components: {nx.number_connected_components(G_similarity):,}")

## 4. DiffusionRank Algorithm

Random walk with restart on the similarity graph to propagate priority scores.

In [ ]:
def diffusion_rank(G, seed_scores, alpha=0.85, max_iter=100, tol=1e-6):
    """
    DiffusionRank: Random walk with restart for priority score propagation.
    
    Args:
        G: NetworkX graph
        seed_scores: Dict of {node: initial_score}
        alpha: Restart probability (higher = more influence from seeds)
        max_iter: Maximum iterations
        tol: Convergence tolerance
    
    Returns:
        Dict of {node: diffusion_score}
    """
    nodes = list(G.nodes())
    n = len(nodes)
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    # Initialize seed vector
    seed_vector = np.zeros(n)
    for node, score in seed_scores.items():
        if node in node_to_idx:
            seed_vector[node_to_idx[node]] = score
    
    # Normalize seed vector
    if seed_vector.sum() > 0:
        seed_vector = seed_vector / seed_vector.sum()
    
    # Build transition matrix
    adj_matrix = nx.to_numpy_array(G, nodelist=nodes, weight='weight')
    row_sums = adj_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # Avoid division by zero
    transition_matrix = adj_matrix / row_sums
    
    # Initialize rank vector
    rank_vector = seed_vector.copy()
    
    # Iterative propagation
    for iteration in range(max_iter):
        rank_vector_new = (1 - alpha) * transition_matrix.T @ rank_vector + alpha * seed_vector
        
        # Check convergence
        diff = np.abs(rank_vector_new - rank_vector).sum()
        if diff < tol:
            print(f"  Converged after {iteration+1} iterations")
            break
        
        rank_vector = rank_vector_new
    
    # Convert back to dict
    scores = {nodes[i]: rank_vector[i] for i in range(n)}
    return scores

In [ ]:
# Create seed scores from baseline model
train_scores = baseline_model.predict(X_train)
seed_scores = dict(zip(train_df['cve_id'], train_scores))

print("Running DiffusionRank...")
diffusion_scores = diffusion_rank(G_similarity, seed_scores, alpha=0.85)

# Add to dataframe
train_df['diffusion_score'] = train_df['cve_id'].map(diffusion_scores)

print(f"\nDiffusion scores computed for {len(diffusion_scores):,} CVEs")
print(f"Score range: [{min(diffusion_scores.values()):.6f}, {max(diffusion_scores.values()):.6f}]")

# For test set, use baseline scores (no graph available)
test_df['diffusion_score'] = test_df['baseline_score']  # Fallback

In [ ]:
# Visualize score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df['baseline_score'], bins=50, alpha=0.7, label='Baseline')
axes[0].hist(train_df['diffusion_score'], bins=50, alpha=0.7, label='DiffusionRank')
axes[0].set_xlabel('Priority Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Score Distribution Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(train_df['baseline_score'], train_df['diffusion_score'], alpha=0.3, s=10)
axes[1].plot([0, 1], [0, 1], 'r--', label='y=x')
axes[1].set_xlabel('Baseline Score')
axes[1].set_ylabel('DiffusionRank Score')
axes[1].set_title('Score Correlation')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

correlation = train_df[['baseline_score', 'diffusion_score']].corr().iloc[0, 1]
print(f"\nCorrelation: {correlation:.4f}")

## 5. Model Comparison

Compare baseline LambdaRank vs DiffusionRank performance.

In [ ]:
# Evaluate DiffusionRank on test set
diffusion_results = {}

for k in k_values:
    ndcg = ndcg_at_k(y_test, test_df['diffusion_score'], k)
    prec = precision_at_k(y_test, test_df['diffusion_score'], k, threshold=3)
    diffusion_results[f'NDCG@{k}'] = ndcg
    diffusion_results[f'Precision@{k}'] = prec

# Create comparison DataFrame
comparison_df = pd.DataFrame({
    'Baseline': baseline_results,
    'DiffusionRank': diffusion_results
})
comparison_df['Improvement'] = ((comparison_df['DiffusionRank'] - comparison_df['Baseline']) / comparison_df['Baseline'] * 100).round(2)
comparison_df['Improvement'] = comparison_df['Improvement'].astype(str) + '%'

print("\nModel Comparison:")
print("=" * 60)
print(comparison_df.to_string())

# Highlight best model per metric
print("\n" + "=" * 60)
for metric in comparison_df.index:
    best_model = comparison_df.loc[metric, ['Baseline', 'DiffusionRank']].idxmax()
    best_value = comparison_df.loc[metric, best_model]
    print(f"{metric:20s}: {best_model:15s} ({best_value:.4f})")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))

metrics = [m for m in comparison_df.index if 'NDCG' in m]
x = np.arange(len(metrics))
width = 0.35

ax.bar(x - width/2, [baseline_results[m] for m in metrics], width, label='Baseline', alpha=0.8)
ax.bar(x + width/2, [diffusion_results[m] for m in metrics], width, label='DiffusionRank', alpha=0.8)

ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('NDCG Comparison: Baseline vs DiffusionRank', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0.85, 1.0])

plt.tight_layout()
plt.show()

## 6. Top-K Analysis

Examine the highest priority CVEs identified by each model.

In [ ]:
# Top-20 CVEs from each model
top_20_baseline = test_df.nlargest(20, 'baseline_score')[[
    'cve_id', 'cvss', 'epss_score', 'kev_flag', 'is_healthcare', 'label', 'baseline_score'
]]

top_20_diffusion = test_df.nlargest(20, 'diffusion_score')[[
    'cve_id', 'cvss', 'epss_score', 'kev_flag', 'is_healthcare', 'label', 'diffusion_score'
]]

print("Top-20 CVEs: Baseline LambdaRank")
print("=" * 80)
print(top_20_baseline.to_string(index=False))

print("\n\nTop-20 CVEs: DiffusionRank")
print("=" * 80)
print(top_20_diffusion.to_string(index=False))

In [ ]:
# Overlap analysis
baseline_top_ids = set(top_20_baseline['cve_id'])
diffusion_top_ids = set(top_20_diffusion['cve_id'])

overlap = baseline_top_ids & diffusion_top_ids
baseline_only = baseline_top_ids - diffusion_top_ids
diffusion_only = diffusion_top_ids - baseline_top_ids

print(f"\nTop-20 Overlap Analysis:")
print(f"  Overlap: {len(overlap)} CVEs")
print(f"  Baseline only: {len(baseline_only)} CVEs")
print(f"  DiffusionRank only: {len(diffusion_only)} CVEs")

if len(diffusion_only) > 0:
    print(f"\nCVEs uniquely identified by DiffusionRank:")
    unique_diffusion = test_df[test_df['cve_id'].isin(diffusion_only)][[
        'cve_id', 'cvss', 'kev_flag', 'is_healthcare', 'label', 'cwe_primary'
    ]]
    print(unique_diffusion.to_string(index=False))

## 7. Summary & Conclusions

Key findings from advanced model experiments.

In [ ]:
# Summary statistics
print("=" * 80)
print("PHASE 7 ADVANCED MODELS - SUMMARY")
print("=" * 80)

print(f"\n1. DATA STATISTICS")
print(f"   Training CVEs: {len(train_df):,}")
print(f"   Test CVEs: {len(test_df):,}")
print(f"   CVEs with CWE: {train_df['has_cwe'].sum():,}")
print(f"   Unique CWEs: {train_df['cwe_primary'].nunique():,}")

print(f"\n2. GRAPH STATISTICS")
print(f"   Bipartite graph edges: {G_bipartite.number_of_edges():,}")
print(f"   Similarity graph edges: {G_similarity.number_of_edges():,}")
print(f"   Avg CVE similarity degree: {sum(dict(G_similarity.degree()).values()) / G_similarity.number_of_nodes():.2f}")

print(f"\n3. PERFORMANCE COMPARISON")
print(f"   Baseline NDCG@20: {baseline_results['NDCG@20']:.4f}")
print(f"   DiffusionRank NDCG@20: {diffusion_results['NDCG@20']:.4f}")
improvement = (diffusion_results['NDCG@20'] - baseline_results['NDCG@20']) / baseline_results['NDCG@20'] * 100
print(f"   Improvement: {improvement:+.2f}%")

print(f"\n4. NEXT STEPS")
print(f"   - Implement RGCN model with PyTorch Geometric")
print(f"   - Create ensemble combining all models")
print(f"   - Add GPU acceleration for training")
print(f"   - Expand graph with CVE-Product relationships")

print("\n" + "=" * 80)
print(f"Notebook completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)